# Memory AI Lab — H2.a : Fine-tuning Bi-encoder

**GPU requis** → `Exécution > Modifier le type d'exécution > GPU T4`

## Principe

mE5-base est générique — il ne sait pas ce qu'est un "épisode" dans notre groupe WhatsApp.
Après fine-tuning contrastif sur nos 448 épisodes gold :

```
Avant : cos_sim(msg_projet_AWS, msg_weekend) ≈ 0.70  (trop proche)
Après : cos_sim(msg_projet_AWS, msg_weekend) ≈ 0.30  (séparés)

Avant : cos_sim(msg_budget_Q3, msg_réunion_lundi) ≈ 0.65
Après : cos_sim(msg_budget_Q3, msg_réunion_lundi) ≈ 0.90  (même épisode)
```

## Loss : MultipleNegativesRankingLoss

```
Batch de paires positives : (msg_a, msg_b) ← même épisode
Négatifs in-batch : tous les autres msg_b du batch
→ Le modèle apprend à maximiser sim(a, b+) vs sim(a, b-)
```

Gain estimé : **+15 à +25% ARI** (tous les stages bénéficient).

## Données requises sur Drive (`memory_ai_data/`)
```
group_anon.txt
group_gold_tune.json
```
Sortie : `memory_ai_data/me5_finetuned/` (modèle HuggingFace)

In [ ]:
# ── CELLULE 1 : Code depuis GitHub ────────────────────────────────────────
import os, sys
REPO = 'https://github.com/Eloekamaje/memory_ai.git'
CODE_DIR = '/content/memory_ai'
if os.path.exists(CODE_DIR):
    !git -C {CODE_DIR} pull --quiet
else:
    !git clone {REPO} {CODE_DIR} --quiet
sys.path.insert(0, f'{CODE_DIR}/src')
print('✓ Code prêt')

In [ ]:
# ── CELLULE 2 : Dépendances ────────────────────────────────────────────────
!pip install "sympy==1.13.1" -q
!pip install -r {CODE_DIR}/requirements_colab.txt -q
!pip install "sentence-transformers>=3.0.0" -q
print('✓ OK')
print()
print('⚠️  Si première exécution : Exécution > Redémarrer la session,')
print('   puis relancer à partir de la cellule 3.')

In [ ]:
# ── CELLULE 3 : Google Drive + Copie locale ────────────────────────────────
import os, shutil, json
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/memory_ai_data'
LOCAL_DIR = '/content/data'
os.makedirs(LOCAL_DIR, exist_ok=True)

for fname in ['group_anon.txt', 'group_gold_tune.json']:
    src, dst = f'{DRIVE_DIR}/{fname}', f'{LOCAL_DIR}/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f'  Copié : {fname} ✓')
    elif os.path.exists(dst):
        print(f'  {fname} déjà en local ✓')
    else:
        print(f'  ⚠️  {fname} absent sur Drive')

DATA_DIR = LOCAL_DIR
print(f'\n✓ DATA_DIR = {DATA_DIR}')

In [ ]:
# ── CELLULE 4 : Charger les données gold et construire les paires ──────────
# Protocole : on utilise TOUT tune pour le bi-encoder
# (différent du TCN qui n'utilise que tune_early)
# Le bi-encoder apprend la métrique de similarité, pas les params de segmentation
import json, random
import numpy as np
from parsers.whatsapp_parser import parse_whatsapp_chat

# Charger les artifacts réels (textes pour les paires)
all_artifacts = parse_whatsapp_chat(f'{DATA_DIR}/group_anon.txt')

# Charger le gold tune
with open(f'{DATA_DIR}/group_gold_tune.json') as f:
    gold = json.load(f)

episodes = gold['episodes']   # [{episode_id, start_idx, end_idx, n_messages}]
artifacts_meta = gold['artifacts']  # [{idx, timestamp, author, content}]

print(f'Episodes gold tune : {len(episodes)}')
print(f'Artifacts tune     : {len(artifacts_meta)}')
print(f'Artifacts parsés   : {len(all_artifacts)}')
print()

# Construire les paires positives (même épisode)
# Pour chaque épisode, échantillonner des paires aléatoires
MAX_PAIRS_PER_EPISODE = 15  # équilibre couverture / surreprésentation
random.seed(42)

positive_pairs = []   # [(text_a, text_b)]

for ep in episodes:
    start, end = ep['start_idx'], ep['end_idx']
    indices = list(range(start, end + 1))

    # Filtrer les messages vides ou trop courts
    valid = [i for i in indices
             if i < len(all_artifacts)
             and len(all_artifacts[i].content.strip()) > 5]

    if len(valid) < 2:
        continue

    # Générer toutes les paires possibles, puis échantillonner
    all_pairs = [(a, b) for i, a in enumerate(valid) for b in valid[i+1:]]
    sampled   = random.sample(all_pairs, min(MAX_PAIRS_PER_EPISODE, len(all_pairs)))

    for i, j in sampled:
        text_a = all_artifacts[i].content.strip()
        text_b = all_artifacts[j].content.strip()
        positive_pairs.append((text_a, text_b))

random.shuffle(positive_pairs)

print(f'Paires positives générées : {len(positive_pairs)}')
print(f'Exemples :')
for a, b in positive_pairs[:3]:
    print(f'  A: {a[:60]}')
    print(f'  B: {b[:60]}')
    print()

In [ ]:
# ── CELLULE 5 : Fine-tuning avec MultipleNegativesRankingLoss ─────────────
# MNRL : paires positives uniquement, négatifs = autres msgs du batch
# → Simple, efficace, recommandé pour les bi-encoders (Reimers & Gurevych 2021)
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')

MODEL_NAME = 'intfloat/multilingual-e5-base'
PREFIX     = 'passage: '   # requis par mE5

model = SentenceTransformer(MODEL_NAME, device=device)

# Préparer les InputExamples (paires positives)
train_examples = [
    InputExample(texts=[PREFIX + a, PREFIX + b])
    for a, b in positive_pairs
]

# DataLoader
BATCH_SIZE = 64   # plus le batch est grand, plus les négatifs sont diversifiés
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)

# Loss MNRL
train_loss = losses.MultipleNegativesRankingLoss(model)

# Fine-tuning
N_EPOCHS   = 5
WARMUP     = int(len(train_dataloader) * 0.1)  # 10% warmup

print(f'Paires       : {len(train_examples)}')
print(f'Batches/epoch: {len(train_dataloader)}')
print(f'Epochs       : {N_EPOCHS}')
print(f'Warmup steps : {WARMUP}')
print()

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=N_EPOCHS,
    warmup_steps=WARMUP,
    optimizer_params={'lr': 2e-5},
    show_progress_bar=True,
)

print('\n✓ Fine-tuning terminé')

In [ ]:
# ── CELLULE 6 : Évaluation qualitative ────────────────────────────────────
# Comparer cos_sim avant/après fine-tuning sur des paires connues
# Positives (même épisode) → sim devrait ↑
# Négatives (épisodes différents) → sim devrait ↓
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Échantillon de test : 50 paires positives + 50 paires négatives
pos_sample = positive_pairs[:50]

# Paires négatives : messages d'épisodes différents
neg_pairs = []
ep_list = [ep for ep in episodes if ep['end_idx'] - ep['start_idx'] >= 1]
for _ in range(50):
    ep_a, ep_b = random.sample(ep_list, 2)
    i = random.randint(ep_a['start_idx'], ep_a['end_idx'])
    j = random.randint(ep_b['start_idx'], ep_b['end_idx'])
    if i < len(all_artifacts) and j < len(all_artifacts):
        neg_pairs.append((all_artifacts[i].content, all_artifacts[j].content))

def eval_pairs(pairs, label):
    texts_a = [PREFIX + a for a, b in pairs]
    texts_b = [PREFIX + b for a, b in pairs]
    emb_a = model.encode(texts_a, convert_to_numpy=True)
    emb_b = model.encode(texts_b, convert_to_numpy=True)
    sims = [cosine_similarity([emb_a[i]], [emb_b[i]])[0][0] for i in range(len(pairs))]
    print(f'  {label:20s} : moy={np.mean(sims):.4f}  std={np.std(sims):.4f}  '
          f'min={np.min(sims):.4f}  max={np.max(sims):.4f}')
    return np.mean(sims)

print('Similarités cosinus (modèle fine-tuné) :')
sim_pos = eval_pairs(pos_sample, 'Même épisode (+)')
sim_neg = eval_pairs(neg_pairs,  'Épisodes diff (-)')
print(f'\n  Séparation : {sim_pos - sim_neg:.4f}  (plus élevé = meilleure discrimination)')
print('  Attendu > 0.10 pour un gain ARI significatif')

In [ ]:
# ── CELLULE 7 : Sauvegarde modèle + génération embeddings ─────────────────
import shutil
from pathlib import Path

# Sauvegarder le modèle fine-tuné
LOCAL_MODEL = '/content/data/me5_finetuned'
DRIVE_MODEL = f'{DRIVE_DIR}/me5_finetuned'

model.save(LOCAL_MODEL)
if os.path.exists(DRIVE_MODEL):
    shutil.rmtree(DRIVE_MODEL)
shutil.copytree(LOCAL_MODEL, DRIVE_MODEL)
print(f'✓ Modèle sauvegardé → Drive:{DRIVE_MODEL}')

# Générer les embeddings complets avec le modèle fine-tuné
texts_all = [PREFIX + a.content for a in all_artifacts]
print(f'\nGénération embeddings ({len(texts_all)} msgs) ...')

emb_finetuned = model.encode(
    texts_all, batch_size=256, show_progress_bar=True,
    device=device, convert_to_numpy=True
).astype(np.float32)

LOCAL_EMB = Path(DATA_DIR) / 'group_embeddings_me5ft.npy'
DRIVE_EMB = f'{DRIVE_DIR}/group_embeddings_me5ft.npy'

np.save(LOCAL_EMB, emb_finetuned)
shutil.copy2(LOCAL_EMB, DRIVE_EMB)
print(f'✓ Embeddings fine-tunés → {LOCAL_EMB.name} + Drive')
print(f'  Shape : {emb_finetuned.shape}')

## Étapes suivantes

Dans **01_eval_ari.ipynb**, changer la cellule 4 :
```python
# Au lieu de group_embeddings_me5.npy :
EMBED_CACHE = Path(DATA_DIR) / 'group_embeddings_me5ft.npy'
# + charger le modèle fine-tuné si recalcul nécessaire :
MODEL_NAME = f'{DATA_DIR}/me5_finetuned'
```

Dans **02_train_boundary_detector.ipynb**, même modification.

Comparer ARI avec baseline 0.9012 → mesure l'impact réel du fine-tuning.